# End-to-End Usage Example
This notebook demonstrates a full workflow for `transform_emr`: install/import, data preparation, train/load (Phase 1 + Phase 2), and evaluation.

In [1]:
%pip install -e .

Obtaining file:///C:/Users/shaha/Work/Personal/Transform-EMR
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for transform_emr (pyproject.toml): started
  Building editable for transform_emr (pyproject.toml): finished with status 'done'
  Created wheel for transform_emr: filename=transform_emr-0.1-0.editable-py3-none-any.whl size=3982 sha256=595c43fe920e0f660413e91f052c8a2af29945e6c1d5cf28787808603c0d10a5
  Stored in directory: C:\Users\shaha\AppData\Local\Temp\pip-ephem-wheel-cache-npfqhdy1\wheels\6d\6c\f9\48dfb1d8b6464f43416

In [2]:
import importlib
import random
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import transform_emr.config.dataset_config as dataset_config
import transform_emr.config.model_config as model_config
from transform_emr import dataset, embedder, transformer, inference

for module in (
    dataset_config,
    model_config,
    dataset,
    embedder,
    transformer,
    inference,
 ):
    importlib.reload(module)

print("Reloaded config and modules. Re-run this cell after changing *_config.py or module code.")
print(f"Project root: {PROJECT_ROOT}")

Reloaded config and modules. Re-run this cell after changing *_config.py or module code.
Project root: C:\Users\shaha\Work\Personal\Transform-EMR


c:\Users\shaha\Work\Personal\Transform-EMR\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Optional cleanup controls for checkpoint artifacts (useful on restricted VMs)
CLEAN_SCALER = False
CLEAN_TOKENIZER = False
CLEAN_EMBEDDER = False
CLEAN_TRANSFORMER = False

# Convenience switch: if True, cleans all artifacts above
CLEAN_ALL = True

clean_scaler = CLEAN_SCALER or CLEAN_ALL
clean_tokenizer = CLEAN_TOKENIZER or CLEAN_ALL
clean_embedder = CLEAN_EMBEDDER or CLEAN_ALL
clean_transformer = CLEAN_TRANSFORMER or CLEAN_ALL

paths_to_delete = []

if clean_scaler:
    paths_to_delete.append(Path(model_config.CHECKPOINT_PATH) / "scaler.pkl")
if clean_tokenizer:
    paths_to_delete.append(Path(model_config.CHECKPOINT_PATH) / "tokenizer.pt")
if clean_embedder:
    embedder_ckpt = Path(model_config.PHASE1_CHECKPOINT).resolve()
    paths_to_delete.extend([embedder_ckpt, embedder_ckpt.parent / "ckpt_last.pt"])
if clean_transformer:
    transformer_ckpt = Path(model_config.PHASE2_CHECKPOINT).resolve()
    paths_to_delete.extend([transformer_ckpt, transformer_ckpt.parent / "ckpt_last.pt"])
    transformer_ckpt = Path(model_config.PHASE3_CHECKPOINT).resolve()
    paths_to_delete.extend([transformer_ckpt, transformer_ckpt.parent / "ckpt_last.pt"])

if not paths_to_delete:
    print("No cleanup selected. Set CLEAN_* = True (or CLEAN_ALL = True) and re-run this cell.")
else:
    removed, missing, failed = [], [], []
    for path in dict.fromkeys(paths_to_delete):  # dedupe while preserving order
        try:
            if path.exists():
                path.unlink()
                removed.append(path)
            else:
                missing.append(path)
        except Exception as exc:
            failed.append((path, str(exc)))

    print("Checkpoint cleanup summary:")
    print(f"  Removed: {len(removed)}")
    for path in removed:
        print(f"    - {path}")

    print(f"  Not found: {len(missing)}")
    for path in missing:
        print(f"    - {path}")

    if failed:
        print(f"  Failed: {len(failed)}")
        for path, err in failed:
            print(f"    - {path} -> {err}")

Checkpoint cleanup summary:
  Removed: 6
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\scaler.pkl
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\tokenizer.pt
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\phase1\ckpt_best.pt
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\phase1\ckpt_last.pt
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\phase2\ckpt_best.pt
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\phase2\ckpt_last.pt
  Not found: 2
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\phase3\ckpt_best.pt
    - C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\phase3\ckpt_last.pt


## 1) Data Load and Processing

In [4]:
# Choose data source mode:
# - "pre_split": use dataset_config TRAIN_* for train and TEST_* for evaluation
# - "source_split": load from data/source and split by PatientId
DATA_SOURCE_MODE = "pre_split"  # "pre_split" | "source_split"

if DATA_SOURCE_MODE not in {"pre_split", "source_split"}:
    raise ValueError("DATA_SOURCE_MODE must be 'pre_split' or 'source_split'.")

# Used only when DATA_SOURCE_MODE == "source_split"
SOURCE_TEMPORAL_DATA_FILE = PROJECT_ROOT / "data" / "source" / "synthetic_diabetes_temporal_data.csv"
SOURCE_CTX_DATA_FILE = PROJECT_ROOT / "data" / "source" / "synthetic_diabetes_context_data.csv"
SOURCE_TEST_SIZE = 0.2
SOURCE_SPLIT_SEED = 42

# Train/val split (applied on the train pool in both modes)
TRAIN_VAL_SIZE = 0.2
TRAIN_VAL_SPLIT_SEED = 42

# Optional train-patient sampling for faster local runs (None = all patients)
SAMPLE_PATIENTS = None
NUM_WORKERS = 4 if SAMPLE_PATIENTS is None else 0  # use single worker if sampling to avoid overhead

# Tokenizer behavior (existing tokenizer is reused if present)
TOKENIZER_PATH = Path(model_config.CHECKPOINT_PATH) / "tokenizer.pt"

# "train_only" | "train_val" | "all_processed" (recommended)
TOKENIZER_FIT_SCOPE = "all_processed"
if TOKENIZER_FIT_SCOPE not in {"train_only", "train_val", "all_processed"}:
    raise ValueError("TOKENIZER_FIT_SCOPE must be 'train_only', 'train_val', or 'all_processed'.")

# Optional safety: auto-sync MODEL_CONFIG ctx_dim from processed train context
AUTO_SET_CTX_DIM = True

# Evaluation settings used later
K_RANGE = range(3, 8)
MAX_LEN = 1000
TEMPERATURE = 1.0
TIME_BIAS_HOURS = 48

print(f"Data mode: {DATA_SOURCE_MODE}")

# Load data according to the selected mode
if DATA_SOURCE_MODE == "pre_split":
    train_temporal_raw = pd.read_csv(dataset_config.TRAIN_TEMPORAL_DATA_FILE, low_memory=False)
    train_ctx_raw = pd.read_csv(dataset_config.TRAIN_CTX_DATA_FILE)
    eval_temporal_raw = pd.read_csv(dataset_config.TEST_TEMPORAL_DATA_FILE, low_memory=False)
    eval_ctx_raw = pd.read_csv(dataset_config.TEST_CTX_DATA_FILE)
else:
    if not SOURCE_TEMPORAL_DATA_FILE.exists() or not SOURCE_CTX_DATA_FILE.exists():
        raise FileNotFoundError(
            f"Source files not found:\n- {SOURCE_TEMPORAL_DATA_FILE}\n- {SOURCE_CTX_DATA_FILE}"
        )

    source_temporal_raw = pd.read_csv(SOURCE_TEMPORAL_DATA_FILE, low_memory=False)
    source_ctx_raw = pd.read_csv(SOURCE_CTX_DATA_FILE)

    source_pids = source_temporal_raw["PatientId"].dropna().unique()
    train_pool_ids, eval_ids = train_test_split(
        source_pids, test_size=SOURCE_TEST_SIZE, random_state=SOURCE_SPLIT_SEED
    )

    train_temporal_raw = source_temporal_raw[source_temporal_raw["PatientId"].isin(train_pool_ids)].copy()
    train_ctx_raw = source_ctx_raw[source_ctx_raw["PatientId"].isin(train_pool_ids)].copy()

    eval_temporal_raw = source_temporal_raw[source_temporal_raw["PatientId"].isin(eval_ids)].copy()
    eval_ctx_raw = source_ctx_raw[source_ctx_raw["PatientId"].isin(eval_ids)].copy()

    # Save the splits for reproducibility and inspection
    train_temporal_raw.to_csv(dataset_config.TRAIN_TEMPORAL_DATA_FILE, index=False)
    train_ctx_raw.to_csv(dataset_config.TRAIN_CTX_DATA_FILE, index=False)
    eval_temporal_raw.to_csv(dataset_config.TEST_TEMPORAL_DATA_FILE, index=False)
    eval_ctx_raw.to_csv(dataset_config.TEST_CTX_DATA_FILE, index=False)

# Optional patient sampling for faster iterations
if SAMPLE_PATIENTS is not None:
    unique_train_pids = train_temporal_raw["PatientId"].dropna().unique()
    if SAMPLE_PATIENTS > len(unique_train_pids):
        raise ValueError(f"SAMPLE_PATIENTS={SAMPLE_PATIENTS} is larger than available patients={len(unique_train_pids)}")
    sampled_ids = sorted(random.sample(list(unique_train_pids), SAMPLE_PATIENTS))
    train_temporal_raw = train_temporal_raw[train_temporal_raw["PatientId"].isin(sampled_ids)].copy()
    train_ctx_raw = train_ctx_raw[train_ctx_raw["PatientId"].isin(sampled_ids)].copy()

# Split train pool into train/val by PatientId
train_pool_pids = train_temporal_raw["PatientId"].dropna().unique()
train_ids, val_ids = train_test_split(
    train_pool_pids, test_size=TRAIN_VAL_SIZE, random_state=TRAIN_VAL_SPLIT_SEED
)

train_temporal_split = train_temporal_raw[train_temporal_raw["PatientId"].isin(train_ids)].copy()
train_ctx_split = train_ctx_raw[train_ctx_raw["PatientId"].isin(train_ids)].copy()
val_temporal_split = train_temporal_raw[train_temporal_raw["PatientId"].isin(val_ids)].copy()
val_ctx_split = train_ctx_raw[train_ctx_raw["PatientId"].isin(val_ids)].copy()

print("Processing train split (fits scaler)...")
train_processor = dataset.DataProcessor(
    train_temporal_split,
    train_ctx_split,
    scaler=None,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
 )
train_temporal_df, train_ctx_df = train_processor.run()

scaler_path = Path(model_config.CHECKPOINT_PATH) / "scaler.pkl"
if not scaler_path.exists():
    raise FileNotFoundError(f"Expected scaler at {scaler_path}")
scaler = joblib.load(scaler_path)

print("Processing val split (uses fitted scaler)...")
val_processor = dataset.DataProcessor(
    val_temporal_split,
    val_ctx_split,
    scaler=scaler,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
 )
val_temporal_df, val_ctx_df = val_processor.run()

print("Processing eval split (uses fitted scaler)...")
eval_processor = dataset.DataProcessor(
    eval_temporal_raw.copy(),
    eval_ctx_raw.copy(),
    scaler=scaler,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
 )
eval_temporal_df, eval_ctx_df = eval_processor.run()

if TOKENIZER_PATH.exists():
    print(f"Loading tokenizer from: {TOKENIZER_PATH}")
    tokenizer = dataset.EMRTokenizer.load(TOKENIZER_PATH)
else:
    if TOKENIZER_FIT_SCOPE == "train_only":
        tokenizer_fit_df = train_temporal_df
    elif TOKENIZER_FIT_SCOPE == "train_val":
        tokenizer_fit_df = pd.concat([train_temporal_df, val_temporal_df], ignore_index=True)
    else:
        tokenizer_fit_df = pd.concat([train_temporal_df, val_temporal_df, eval_temporal_df], ignore_index=True)

    print(
        "Building tokenizer from "
        f"{TOKENIZER_FIT_SCOPE} data: "
        f"{tokenizer_fit_df['PatientId'].nunique()} patients, "
        f"{tokenizer_fit_df['PositionToken'].nunique()} unique position tokens"
    )
    tokenizer = dataset.EMRTokenizer.from_processed_df(tokenizer_fit_df)
    tokenizer.save(TOKENIZER_PATH)
    print(f"Saved tokenizer to: {TOKENIZER_PATH}")

train_ds = dataset.EMRDataset(train_temporal_df, train_ctx_df, tokenizer=tokenizer)
val_ds = dataset.EMRDataset(val_temporal_df, val_ctx_df, tokenizer=tokenizer)
eval_ds = dataset.EMRDataset(eval_temporal_df, eval_ctx_df, tokenizer=tokenizer)

if AUTO_SET_CTX_DIM:
    model_config.MODEL_CONFIG["ctx_dim"] = int(train_ds.context_df.shape[1])
    print(f"Updated MODEL_CONFIG['ctx_dim'] -> {model_config.MODEL_CONFIG['ctx_dim']}")
else:
    assert train_ds.context_df.shape[1] == model_config.MODEL_CONFIG["ctx_dim"], (
        f"Context dimension mismatch: expected {model_config.MODEL_CONFIG['ctx_dim']}, got {train_ds.context_df.shape[1]}"
    )

embedder_train_dl = dataset.get_dataloader(
    train_ds,
    batch_size=model_config.TRAINING_SETTINGS["batch_size"],
    collate_fn=dataset.collate_emr,
    oversample=False,
    num_workers=NUM_WORKERS,
 )
transformer_train_dl = dataset.get_dataloader(
    train_ds,
    batch_size=model_config.TRAINING_SETTINGS["batch_size"],
    collate_fn=dataset.collate_emr,
    oversample=True,
    num_workers=NUM_WORKERS,
 )
val_dl = dataset.get_dataloader(
    val_ds,
    batch_size=model_config.TRAINING_SETTINGS["batch_size"],
    collate_fn=dataset.collate_emr,
    oversample=False,
    num_workers=NUM_WORKERS,
 )

print(f"Train patients: {len(train_ids)} | Val patients: {len(val_ids)} | Eval patients: {eval_temporal_df['PatientId'].nunique()}")
print(f"Train records: {len(train_ds.tokens_df):,} | Val records: {len(val_ds.tokens_df):,} | Eval records: {len(eval_ds.tokens_df):,}")

Data mode: pre_split
Processing train split (fits scaler)...
Processing val split (uses fitted scaler)...
Processing eval split (uses fitted scaler)...
Building tokenizer from all_processed data: 500 patients, 386 unique position tokens
Saved tokenizer to: C:\Users\shaha\Work\Personal\Transform-EMR\checkpoints\tokenizer.pt
Updated MODEL_CONFIG['ctx_dim'] -> 2
Train patients: 400 | Val patients: 100 | Eval patients: 100
Train records: 34,045 | Val records: 6,418 | Eval records: 8,485


## 2) Training

In [5]:
# ─── Training Control Panel ──────────────────────────────────────────────
#
# RUN_PHASE1 : Whether to run Phase-1 (embedder) training.
#              True  -> train the embedder (fresh or resumed per RESUME_TRAINING).
#              False -> skip; best available Phase-1 checkpoint is loaded instead.
#
# RUN_PHASE2 : Whether to run Phase-2 (transformer) training.
#              True  -> train the transformer (fresh or resumed per RESUME_TRAINING).
#              False -> skip; best available Phase-2 checkpoint is loaded instead.
#
# RUN_PHASE3 : Whether to run Phase-3 (outcome head alignment).
#              True  -> fine-tune outcome_head on free-running trajectories.
#              False -> skip; loads Phase-3 checkpoint if available, otherwise uses
#                      Phase-2 outcome head (may have teacher-forcing distribution gap).
#
# RESUME_TRAINING : Whether each active phase continues from its last checkpoint.
#              True  -> restore weights + optimizer + scheduler + aux-scheduler and continue.
#              False -> start that phase from epoch 0 with fresh optimizer/scheduler state.
#              (Has no effect on phases that are skipped -- those always load best checkpoint.)
#
# EVAL_INPUT_DAYS : Number of days of input context used when building the truncated
#              eval dataset for generate_risk_curves (Section 3b). Has no effect on
#              Phase-3 training, which uses the standard train DataLoader.
#
# Common configurations:
#   Full fresh start             RUN_PHASE1=True,  RUN_PHASE2=True,  RUN_PHASE3=True,  RESUME_TRAINING=False
#   Resume all phases            RUN_PHASE1=True,  RUN_PHASE2=True,  RUN_PHASE3=True,  RESUME_TRAINING=True
#   Skip phase 1, fresh ph.2-3   RUN_PHASE1=False, RUN_PHASE2=True,  RUN_PHASE3=True,  RESUME_TRAINING=False
#   Phase 3 only (post-training) RUN_PHASE1=False, RUN_PHASE2=False, RUN_PHASE3=True,  RESUME_TRAINING=True
#   Evaluate only (no training)  RUN_PHASE1=False, RUN_PHASE2=False, RUN_PHASE3=False, RESUME_TRAINING=False
# ──────────────────────────────────────────────────────────────────────────────

RUN_PHASE1      = True
RUN_PHASE2      = True
RUN_PHASE3      = True
RESUME_TRAINING = False

EVAL_INPUT_DAYS = 2   # days of patient history used as input for Phase-3 generation

print(f"RUN_PHASE1={RUN_PHASE1}  |  RUN_PHASE2={RUN_PHASE2}  |  RUN_PHASE3={RUN_PHASE3}  |  RESUME_TRAINING={RESUME_TRAINING}")
print(f"EVAL_INPUT_DAYS={EVAL_INPUT_DAYS}")


RUN_PHASE1=True  |  RUN_PHASE2=True  |  RUN_PHASE3=True  |  RESUME_TRAINING=False
EVAL_INPUT_DAYS=2


In [ ]:
def _load_embedder_checkpoint(best_path, tokenizer):
    """Load best Phase-1 checkpoint, falling back to last if best is not yet saved."""
    ckpt_best = Path(best_path).resolve()
    ckpt_last = ckpt_best.parent / "ckpt_last.pt"
    ckpt = ckpt_best if ckpt_best.exists() else ckpt_last
    if not ckpt.exists():
        raise FileNotFoundError(
            "No Phase-1 checkpoint found. Run with RUN_PHASE1=True first."
        )
    print(f"Loading embedder from: {ckpt}")
    model, *_ = embedder.EMREmbedding.load(ckpt, tokenizer=tokenizer)
    return model

def _load_transformer_checkpoint(best_path, embedder_model):
    """Load best Phase-2 checkpoint, falling back to last if best is not yet saved."""
    ckpt_best = Path(best_path).resolve()
    ckpt_last = ckpt_best.parent / "ckpt_last.pt"
    ckpt = ckpt_best if ckpt_best.exists() else ckpt_last
    if not ckpt.exists():
        raise FileNotFoundError(
            "No Phase-2 checkpoint found. Run with RUN_PHASE2=True first."
        )
    print(f"Loading transformer from: {ckpt}")
    model, *_ = transformer.GPT.load(ckpt, embedder=embedder_model)
    return model


# ─── Phase 1: embedder ────────────────────────────────────────────────
# RUN_PHASE1=True : always start with a fresh embedder instance.
#   train_embedder() handles checkpoint loading internally when RESUME_TRAINING=True,
#   restoring weights, optimizer, scheduler, and aux-scheduler state.
# RUN_PHASE1=False: load the best available Phase-1 checkpoint directly.

if RUN_PHASE1:
    embedder_model = embedder.EMREmbedding(
        tokenizer=tokenizer,
        ctx_dim=model_config.MODEL_CONFIG.get("ctx_dim"),
        time2vec_dim=model_config.MODEL_CONFIG.get("time2vec_dim"),
        embed_dim=model_config.MODEL_CONFIG.get("embed_dim"),
    )
    embedder_model, _, _ = embedder.train_embedder(
        embedder=embedder_model,
        train_loader=embedder_train_dl,
        val_loader=val_dl,
        resume=RESUME_TRAINING,
        checkpoint_path=model_config.PHASE1_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    embedder_model = _load_embedder_checkpoint(model_config.PHASE1_CHECKPOINT, tokenizer)


# ─── Phase 2: transformer ──────────────────────────────────────────────
# RUN_PHASE2=True : always initialize a fresh GPT instance wrapping the embedder above.
#   pretrain_transformer() handles checkpoint loading internally when RESUME_TRAINING=True.
#   The fresh GPT passed in is discarded on resume -- only its architecture is used.
# RUN_PHASE2=False: load the best available Phase-2 checkpoint directly.

if RUN_PHASE2:
    model = transformer.GPT(cfg=model_config.MODEL_CONFIG, embedder=embedder_model)
    model, _, _ = transformer.pretrain_transformer(
        model=model,
        train_dl=transformer_train_dl,
        val_dl=val_dl,
        resume=RESUME_TRAINING,
        checkpoint_path=model_config.PHASE2_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    model = _load_transformer_checkpoint(model_config.PHASE2_CHECKPOINT, embedder_model)


# ─── Phase 3: outcome head fine-tuning ─────────────────────────────────────
# Freezes the backbone and fine-tunes only outcome_head on teacher-forced data
# (same DataLoaders as Phase 2). Uses get_future_outcome_targets — identical
# soft-label formula as Phase 2 — but with gradient isolation on the head only.
# RUN_PHASE3=False: loads Phase-3 checkpoint if available (full GPT.load()),
#   otherwise the Phase-2 outcome head is used as-is.

if RUN_PHASE3:
    model, _, _ = transformer.finetune_transformer(
        model=model,
        train_dl=transformer_train_dl,
        val_dl=val_dl,
        resume=RESUME_TRAINING,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    p3_ckpt = Path(model_config.PHASE3_CHECKPOINT)
    p3_last = p3_ckpt.parent / "ckpt_last.pt"
    p3_path = p3_last if p3_last.exists() else (p3_ckpt if p3_ckpt.exists() else None)
    if p3_path:
        print(f"Loading Phase-3 checkpoint from: {p3_path}")
        model, *_ = transformer.GPT.load(p3_path, embedder=embedder_model)
    else:
        print("No Phase-3 checkpoint found; using outcome head from Phase-2.")

model.eval()
print("Model ready for evaluation.")


[Phase-1] Epoch 001
            --> Train=5.3951 (BCE=5.3951  MLM=0.0000  Δt=0.0000)
            --> Val=3.4037 (BCE=3.4037  MLM=0.0000  Δt=0.0000)


Training:   0%|          | 0/7 [00:00<?, ?it/s]

## 3) Risk-Based Complication Prediction

Generates a single trajectory per patient (K=1, autoregressive) and reads the outcome head at every step.
This produces a **risk curve** per complication over predicted time.

Evaluation uses **time-stratified AUC**: at each 24 h window the max predicted probability within that
window is compared against whether the complication occurs within that **same 24 h window** in the ground truth.
AUROC and AUPRC are computed per complication then averaged across time windows.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

from transform_emr.inference import generate_risk_curves
from transform_emr.config import dataset_config

OUTCOME_NAMES = model.outcome_names   # list of outcome token strings
P_COLS        = [f'P_{n}' for n in OUTCOME_NAMES]
DEVICE        = next(model.parameters()).device

### 3a) Ground Truth Extraction

In [ ]:
def extract_ground_truth(dataset):
    """
    For each patient return a dict  outcome_name -> first observed time (hours
    from admission), or np.inf if never observed in the dataset.
    """
    outcome_set = set(OUTCOME_NAMES)
    gt = {}
    for pid in dataset.patient_ids:
        df = dataset.patient_groups[pid]
        patient_gt = {n: np.inf for n in OUTCOME_NAMES}
        tok_col = 'PositionToken' if 'PositionToken' in df.columns else 'Token'
        for _, row in df.iterrows():
            tok_str = row[tok_col]
            if tok_str in outcome_set:
                t = row['TimePoint'] * 336.0  # denormalise
                if t < patient_gt[tok_str]:
                    patient_gt[tok_str] = t
        gt[pid] = patient_gt
    return gt

# dataset_test has full (untruncated) sequences -> use for ground truth
gt_labels = extract_ground_truth(eval_ds)
print(f'Ground truth extracted for {len(gt_labels)} patients.')

import pandas as pd
summary = {n: sum(1 for p in gt_labels if gt_labels[p][n] < np.inf) for n in OUTCOME_NAMES}
pd.Series(summary, name='patients_with_complication').sort_values(ascending=False)

### 3b) Generate Risk Curves

In [ ]:
# Build truncated eval dataset using the same input-day cutoff as Phase-3 training.
# This ensures the outcome head sees the same context distribution it was aligned on.
print(f"Building truncated eval dataset ({EVAL_INPUT_DAYS}-day input window)...")
eval_input_processor = dataset.DataProcessor(
    eval_temporal_raw.copy(),
    eval_ctx_raw.copy(),
    scaler=scaler,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
    max_input_days=EVAL_INPUT_DAYS,
)
eval_input_temporal_df, eval_input_ctx_df = eval_input_processor.run()
eval_ds_input = dataset.EMRDataset(eval_input_temporal_df, eval_input_ctx_df, tokenizer=tokenizer)

model.eval()
risk_df = inference.generate_risk_curves(
    model, eval_ds_input,
    max_len=500,
    temperature=1.0,
    top_k=None,
    rep_decay=0.6,
)
print(f"Risk curves: {len(risk_df)} rows, {risk_df["PatientId"].nunique()} patients.")
risk_df.head()


### 3c) Time-Stratified AUC

At each 24 h window along the generated trajectory:
- **score** = max outcome-head probability within the window (per patient)
- **label** = 1 if the complication first occurs *inside* this same window in the ground truth

AUROC and AUPRC are computed across patients at each window, then averaged.
This directly answers: *'does the model flag the right 24 h periods?'*

In [ ]:
def time_stratified_auc(risk_df, gt_labels, outcome_names,
                         window_hours=24.0, min_pos=3):
    """
    At each 24 h window along the generated trajectory:
      score  = max outcome-head probability within the window (per patient)
      label  = 1 if the complication first occurs inside this same window
               in the ground truth.
    AUROC and AUPRC are computed across patients at each window, then averaged.

    Returns a DataFrame indexed by outcome with columns:
        auroc_mean, auroc_std, auprc_mean, auprc_std, n_windows
    """
    gen_df = risk_df[risk_df['IsInput'] == 0].copy()
    pids   = gen_df['PatientId'].unique()

    t_min  = gen_df['TimePoint'].min()
    t_max  = gen_df['TimePoint'].max()
    time_marks = np.arange(t_min + window_hours, t_max + window_hours, window_hours)

    per_auroc = {n: [] for n in outcome_names}
    per_auprc = {n: [] for n in outcome_names}

    for t_mark in time_marks:
        window_df = gen_df[
            (gen_df['TimePoint'] > t_mark - window_hours) &
            (gen_df['TimePoint'] <= t_mark)
        ]
        if window_df.empty:
            continue

        for name in outcome_names:
            pcol = f'P_{name}'
            scores, labels = [], []
            for pid in pids:
                sub = window_df[window_df['PatientId'] == pid]
                if sub.empty:
                    continue
                score = sub[pcol].max()  # peak risk in this window
                gt_t  = gt_labels.get(pid, {}).get(name, np.inf)
                # label = did this complication occur IN this same window?
                label = int(gt_t > t_mark - window_hours and gt_t <= t_mark)
                scores.append(score)
                labels.append(label)

            labels_arr = np.array(labels)
            if labels_arr.sum() < min_pos or (1 - labels_arr).sum() < min_pos:
                continue
            scores_arr = np.array(scores)
            per_auroc[name].append(roc_auc_score(labels_arr, scores_arr))
            per_auprc[name].append(average_precision_score(labels_arr, scores_arr))

    rows = []
    for name in outcome_names:
        auroc_v = per_auroc[name]
        auprc_v = per_auprc[name]
        rows.append({
            'outcome':    name,
            'auroc_mean': np.mean(auroc_v) if auroc_v else np.nan,
            'auroc_std':  np.std(auroc_v)  if auroc_v else np.nan,
            'auprc_mean': np.mean(auprc_v) if auprc_v else np.nan,
            'auprc_std':  np.std(auprc_v)  if auprc_v else np.nan,
            'n_windows':  len(auroc_v),
        })
    return pd.DataFrame(rows).set_index('outcome').sort_values('auroc_mean', ascending=False)


auc_results = time_stratified_auc(
    risk_df, gt_labels, OUTCOME_NAMES,
    window_hours=24.0,
)
print(f'Mean AUROC across complications: {auc_results["auroc_mean"].mean():.3f}')
print(f'Mean AUPRC across complications: {auc_results["auprc_mean"].mean():.3f}')
auc_results.round(3)

### 3c-ii) Time Accuracy (MAE on Onset Prediction)

For patients where a complication actually occurred, compare the **predicted onset time**
(TimePoint of the generated step where outcome probability peaked) against the **actual onset time**.
Reports Mean Absolute Error in hours per complication.

In [ ]:
def time_accuracy(risk_df, gt_labels, outcome_names):
    """
    For each complication, for patients where it occurred:
      predicted onset = TimePoint of the generated step with peak P_<outcome>
      actual onset    = first occurrence time from ground truth
    Returns MAE in hours per complication.
    """
    gen_df = risk_df[risk_df['IsInput'] == 0].copy()
    rows = []
    for name in outcome_names:
        pcol = f'P_{name}'
        errors = []
        for pid in risk_df['PatientId'].unique():
            gt_t = gt_labels.get(pid, {}).get(name, np.inf)
            if gt_t == np.inf:
                continue  # complication never occurred for this patient
            sub = gen_df[gen_df['PatientId'] == pid]
            if sub.empty:
                continue
            predicted_t = sub.loc[sub[pcol].idxmax(), 'TimePoint']
            errors.append(abs(predicted_t - gt_t))
        rows.append({
            'outcome':   name,
            'mae_hours': np.mean(errors) if errors else np.nan,
            'n_patients': len(errors),
        })
    return pd.DataFrame(rows).set_index('outcome').sort_values('mae_hours')


time_acc = time_accuracy(risk_df, gt_labels, OUTCOME_NAMES)
print(f'Mean MAE across complications with occurrences: {time_acc["mae_hours"].mean():.1f} h')
time_acc.round(1)

### 3d) Temperature Scaling Calibration

Learns one temperature scalar `T` per outcome by minimising NLL on the test set.
Using `logits / T` before sigmoid shrinks (T > 1) or sharpens (T < 1) probabilities
without changing rank order — so AUROC is unaffected, but calibration curves improve.

**When you need this**: when you want to interpret probability values (e.g. 'this patient
has a 70% chance of hyperglycemia') rather than just rank patients by risk.

In [ ]:
def calibrate_temperature(model, dataset, gt_labels, outcome_names,
                           n_iter=200, lr=0.05):
    """
    Teacher-forced forward pass on the full (untruncated) test sequences.
    For each patient, takes the outcome-head logit at the *last real token*
    as the pre-calibration score, then learns T per outcome via LBFGS.

    Returns
    -------
    temperatures : dict  outcome_name -> float T
    cal_data     : dict  outcome_name -> (logits_np, labels_np)
    """
    device = next(model.parameters()).device
    tok    = model.embedder.tokenizer

    all_logits = {n: [] for n in outcome_names}
    all_labels = {n: [] for n in outcome_names}

    model.eval()
    with torch.no_grad():
        for pid in tqdm(dataset.patient_ids, desc='Collecting logits'):
            df      = dataset.patient_groups[pid]
            ctx_vec = torch.tensor(dataset.context_df.loc[pid].values,
                                   dtype=torch.float32).unsqueeze(0).to(device)
            pos_ids        = torch.tensor([df['PositionID'].tolist()], dtype=torch.long, device=device)
            parent_raw_ids = tok.tokenid2parent_raw_ids[pos_ids[0]].unsqueeze(0).to(device)
            concept_ids    = torch.tensor([df['ConceptID'].tolist()],  dtype=torch.long, device=device)
            value_ids      = torch.tensor([df['ValueID'].tolist()],    dtype=torch.long, device=device)
            abs_ts         = torch.tensor([df['TimePoint'].tolist()],  dtype=torch.float32, device=device) / 336.0

            _, _, outcome_logits, _ = model(
                parent_raw_ids=parent_raw_ids, concept_ids=concept_ids,
                value_ids=value_ids, position_ids=pos_ids,
                abs_ts=abs_ts, context_vec=ctx_vec
            )
            last_logit = outcome_logits[0, -1, :].cpu()  # [num_outcomes]

            for i, name in enumerate(outcome_names):
                gt_t  = gt_labels.get(pid, {}).get(name, np.inf)
                label = int(gt_t < np.inf)  # ever occurred?
                all_logits[name].append(last_logit[i].item())
                all_labels[name].append(label)

    temperatures = {}
    cal_data     = {}
    for name in outcome_names:
        logits_t = torch.tensor(all_logits[name], dtype=torch.float32)
        labels_t = torch.tensor(all_labels[name], dtype=torch.float32)
        if labels_t.sum() < 2:
            temperatures[name] = 1.0
            cal_data[name] = (logits_t.numpy(), labels_t.numpy())
            continue
        T = nn.Parameter(torch.ones(1))
        opt = torch.optim.LBFGS([T], lr=lr, max_iter=n_iter)
        def closure():
            opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(logits_t / T, labels_t)
            loss.backward()
            return loss
        opt.step(closure)
        temperatures[name] = T.item()
        cal_data[name] = (logits_t.detach().numpy(), labels_t.numpy())

    return temperatures, cal_data


# calibrate on the FULL test sequences (eval_ds)
temperatures, cal_data = calibrate_temperature(model, eval_ds, gt_labels, OUTCOME_NAMES)
pd.Series(temperatures, name='temperature_T').sort_values()

### 3e) Reliability Diagram (before / after calibration)

In [ ]:
def reliability_diagram(ax, logits, labels, T=1.0, n_bins=10, label=''):
    probs = torch.sigmoid(torch.tensor(logits) / T).numpy()
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers, mean_true = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() > 0:
            bin_centers.append(probs[mask].mean())
            mean_true.append(labels[mask].mean())
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
    ax.plot(bin_centers, mean_true, 'o-', label=label)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

# Plot outcomes that have enough positives
eligible = [n for n in OUTCOME_NAMES if cal_data[n][1].sum() >= 5]
n_plots  = min(len(eligible), 8)
fig, axes = plt.subplots(2, (n_plots + 1) // 2, figsize=(4 * ((n_plots + 1) // 2), 8))
axes = axes.flatten()
for ax, name in zip(axes, eligible[:n_plots]):
    logits_np, labels_np = cal_data[name]
    ax.set_title(name.replace('_EVENT', '').replace('_COMPLICATION', ''), fontsize=8)
    reliability_diagram(ax, logits_np, labels_np, T=1.0,            label='Uncalibrated')
    reliability_diagram(ax, logits_np, labels_np, T=temperatures[name], label='Calibrated')
for ax in axes[n_plots:]:
    ax.set_visible(False)
plt.tight_layout()
plt.suptitle('Reliability Diagrams (outcome head)', y=1.01, fontsize=11)
plt.show()

### 3f) Sample Patient Risk Curve

In [ ]:
# Visualise the risk trajectory for one patient over time.
# Change `sample_pid` to inspect a specific patient.
sample_pid = risk_df['PatientId'].iloc[0]
patient_risk = risk_df[risk_df['PatientId'] == sample_pid].copy()

# Outcomes that actually have some signal for this patient
gen_risk = patient_risk[patient_risk['IsInput'] == 0]
active = [n for n in OUTCOME_NAMES if gen_risk[f'P_{n}'].max() > 0.05]
if not active:
    active = OUTCOME_NAMES[:4]  # fallback: show first four

fig, ax = plt.subplots(figsize=(12, 4))
for name in active:
    ax.plot(patient_risk['TimePoint'], patient_risk[f'P_{name}'], label=name.replace('_EVENT',''), lw=1.5)

# Mark where input ends
input_end = patient_risk[patient_risk['IsInput'] == 1]['TimePoint'].max()
ax.axvline(input_end, color='gray', linestyle='--', lw=1, label='Generation start')

# Mark actual ground truth occurrences (if patient in gt_labels)
if sample_pid in gt_labels:
    for name, t in gt_labels[sample_pid].items():
        if t < np.inf and name in active:
            ax.axvline(t, color='red', linestyle=':', lw=1)

ax.set_xlabel('Hours from admission')
ax.set_ylabel('Outcome probability')
ax.set_title(f'Risk curves — patient {sample_pid}')
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()